# multi_heatmap · 01 — Conformación y exploración de datos

Notebook de **Google Colab** para la exploración de la variante `multi_heatmap`: descarga de eegbci, construcción del dataset multi-referencia, insumos multi-configuración balanceados (19/64/128/256 electrodos) y verificación visual de la física.

**Uso:** Runtime → Change runtime type → **GPU (T4)**, luego ejecutar las celdas en orden.

## 0 · Entorno

In [ ]:
# ---- 0 · Entorno (Colab o local) --------------------------------------
# Colab: Runtime > Change runtime type > GPU (T4). El repo se clona en /content.
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/sonoAESS/universal-eeg-transformer.git"
BRANCH   = "explore/multi-heatmap"

IN_COLAB = "google.colab" in sys.modules or "/content" in os.getcwd()

if IN_COLAB:
    ROOT = Path("/content/universal-eeg-transformer")
    if not ROOT.exists():
        !git clone -b {BRANCH} {REPO_URL} {ROOT}
    # Dependencias: TF ya viene preinstalado; el resto lo trae pyproject vía pip.
    %pip install -q mne pyyaml pandas matplotlib scikit-learn
else:
    ROOT = Path.cwd()
    while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
print(f"ROOT = {ROOT}\nColab = {IN_COLAB}")

In [ ]:
# ---- 0b · Persistencia en Google Drive (opcional) ---------------------
# Cachea dataset + checkpoints en Drive para no re-descargar en cada sesión.
USE_DRIVE = IN_COLAB

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/universal_eeg_cache")
    CACHE.mkdir(exist_ok=True)
    for link in ("data/processed", "runs"):
        target = CACHE / link.split("/")[-1]
        target.mkdir(exist_ok=True)
        dest = ROOT / link
        if not dest.exists():
            dest.symlink_to(target)
    print("Cache y runs enlazados a:", CACHE)
else:
    print("Modo local: cache en ./data/processed y ./runs")

## 1 · Configuración y carga del experimento

In [ ]:
# ---- 1 · Configuración de la variante ---------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# Variante en exploración. Cambiar a config/multi_heatmap_v2.yaml cuando toque.
CONFIG = ROOT / "config/multi_heatmap.yaml"

from eeg_transform.nb import load_experiment
from eeg_transform.experiments.multi import multiconfig_summary, KINDS

cfg, ds = load_experiment(CONFIG)
data = multiconfig_data(cfg, ds)

print(ds.summary())
print()
print(multiconfig_summary(data))
print("\nSplits (budget balanceado):",
      {s: data.n_budget(s) for s in ("train", "val", "test")})

## 2 · Geometría

In [ ]:
# ---- 2 · Geometría: posiciones de electrodos por configuración --------
# Todas las configuraciones viven sobre el mismo casquete esférico; los
# subconjuntos (10-20) son selección exacta de columnas del canónico y los
# densos (128/256) interpolaciones cuasi-uniformes del campo REST.
fig, axes = plt.subplots(1, len(data.order), figsize=(4 * len(data.order), 4))
for ax, label in zip(np.atleast_1d(axes), data.order):
    mc = data[label]
    pos = mc.positions
    ax.scatter(pos[:, 0], pos[:, 1], c=pos[:, 2], s=14, cmap="viridis")
    ax.set_title(f"{label} ({mc.n_channels} ch)")
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Configuraciones de electrodos (proyección XY, color = elevación z)")
plt.tight_layout(); plt.show()

## 3 · Señales

In [ ]:
# ---- 3 · Las cuatro referencias (trazas reales, split test) -----------
# Un segmento del canónico: unipolar es potencial a infinito menos Cz;
# bipolar resta canales vecinos; CAR resta la media; REST usa el lead field.
i0, n_show, ch = 100, 400, 10
t = np.arange(n_show) / cfg.dataset.fs if hasattr(cfg.dataset, "fs") else np.arange(n_show)

refs_test = {k: ds.refs[k][ds.split_idx["test"]] for k in KINDS}
names = list(ds.ch_names)

fig, axes = plt.subplots(len(KINDS), 1, figsize=(12, 9), sharex=True)
for ax, k in zip(axes, KINDS):
    sig = refs_test[k][i0 : i0 + n_show, ch]
    ax.plot(sig, lw=0.8)
    ax.set_ylabel(k); ax.grid(alpha=0.3)
axes[0].set_title(f"Canal '{names[ch]}' — 4 referencias (test)")
plt.xlabel("muestra"); plt.tight_layout(); plt.show()

## 4 · Relación entre referencias

In [ ]:
# ---- 4 · Relación entre referencias -----------------------------------
# Correlación global entre referencias (todas comparten el mismo potencial
# subyacente; difieren solo en la proyección lineal aplicada).
flat = {k: refs_test[k][:2000].ravel() for k in KINDS}
C = np.corrcoef([flat[k] for k in KINDS])

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(C, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(KINDS)), KINDS, rotation=45)
ax.set_yticks(range(len(KINDS)), KINDS)
for a in range(len(KINDS)):
    for b in range(len(KINDS)):
        ax.text(b, a, f"{C[a, b]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, label="r")
ax.set_title("Correlación entre referencias (test)")
plt.tight_layout(); plt.show()

## 5 · Operadores

In [ ]:
# ---- 5 · Operadores de referencia (matrices M) ------------------------
# Convención del repo: X_ref = X @ M (aplicación POR FILAS).
# Cada configuración tiene sus propios operadores sobre su lead field.
from eeg_transform.references import build_reference_matrix

label = "canonical"
mc = data[label]
ops = {
    k: build_reference_matrix(
        k, mc.n_channels,
        unipolar_ref_index=mc.unipolar_ref_index,
        lead_field=mc.leadfield,
        rest_rcond=cfg.leadfield.rest_rcond,
    )
    for k in KINDS
}

fig, axes = plt.subplots(1, len(KINDS), figsize=(4 * len(KINDS), 4.2))
for ax, k in zip(axes, KINDS):
    im = ax.imshow(ops[k], cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(k); ax.set_xlabel("canal entrada"); ax.set_ylabel("canal salida")
fig.suptitle(f"Matrices de referencia ({label}, {mc.n_channels}×{mc.n_channels})")
plt.tight_layout(); plt.show()

print("Rango bipolar (esperado C-1):", np.linalg.matrix_rank(ops["bipolar"]))

## 6 · Campo de superficie

In [ ]:
# ---- 6 · Campo de superficie por configuración ------------------------
# La matriz fija S_s (mc.surface: grid_px² × C_s) interpola electrodos ->
# malla compartida. Es la MISMA malla para todas las configuraciones: aquí
# se verifica que el campo observado coincide entre configs (base del término
# xconfig_consistency de la variante).
from eeg_transform.mapping import scalp_grid_matrix

grid_px = cfg.mapping.grid_px
split, kind, i0 = "test", "unipolar", 42

fig, axes = plt.subplots(2, len(data.order), figsize=(3.4 * len(data.order), 7))
for col, label in enumerate(data.order):
    mc = data[label]
    field = (mc.refs[split][kind][i0] @ mc.surface.T).reshape(grid_px, grid_px)
    axes[0, col].imshow(field, cmap="RdBu_r", origin="lower")
    axes[0, col].set_title(label)
    # diferencia contra el canónico (misma malla -> comparable píxel a píxel)
    canon = data["canonical"].refs[split][kind][i0]
    canon_field = (canon @ data["canonical"].surface.T).reshape(grid_px, grid_px)
    diff = field - canon_field
    im = axes[1, col].imshow(diff, cmap="RdBu_r", origin="lower")
    axes[1, col].set_title(f"{label} − canonical\nmax|Δ|={np.abs(diff).max():.3g}")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"Campo de superficie '{kind}' (test, muestra {i0}): arriba = campo, abajo = Δ vs canónico")
plt.tight_layout(); plt.show()

## 7 · Balanceo

In [ ]:
# ---- 7 · Balanceo y presupuesto ----------------------------------------
# Todas las configuraciones comparten exactamente los mismos índices de
# split: cada una aporta la misma fracción de datos por lote.
rows = []
for label in data.order:
    mc = data[label]
    row = {"config": label, "canales": mc.n_channels}
    for s in ("train", "val", "test"):
        row[f"n_{s}"] = mc.refs[s]["unipolar"].shape[0]
    rows.append(row)
df_budget = pd.DataFrame(rows)
assert all((df_budget[f'n_{s}'] == df_budget['n_train']).all() for s in ("val", "test"))
df_budget

## Conclusiones de la exploración

* Las configuraciones observan **el mismo campo subyacente** (ancla REST
  canónica) en distintas geometrías; las diferencias entre campos son solo
  interpolación/selección espacial.
* Las cuatro referencias son transformaciones lineales del mismo potencial:
  correlación altísima pero matrices `M` de estructura distinta (el bipolar
  pierde el modo constante; el rango es C−1).
* La malla compartida (`grid_px²`) hace los campos comparables píxel a píxel
  entre configuraciones: es la clave del entrenamiento `multi_heatmap`.

Siguiente paso: `02_modelo_entrenamiento.ipynb` construye, entrena y valida
el autoencoder sobre estos insumos.